## 1. Load questions

In [7]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src/app.py").exists():
    ROOT = ROOT / "midterm-project"
sys.path.insert(0, str(ROOT / "src"))
import app

DATASET = ROOT / "data/evaluation_set.json"
dataset = json.loads(DATASET.read_text())
questions = dataset["questions"]
source_index = {text: i for i, text in enumerate(app.DOCUMENTS)}
print(f"Loaded {len(questions)} questions. No API calls made.")
pd.DataFrame(questions)[["id", "type", "question"]].head()


Loaded 24 questions. No API calls made.


,id,type,question
0,Q01,supported,"A country has enough food in its markets, yet ..."
1,Q02,supported,Compare cash assistance and in-kind food assis...
2,Q03,supported,How do vouchers differ from cash transfers in ...
3,Q04,supported,Compare anticipatory action with a humanitaria...
4,Q05,supported,Would increasing calorie availability alone ad...


## 2. Choose the experiments

In [8]:
REPEATS = 3
PROMPT_A = app.SYSTEM_PROMPT
PROMPT_B = PROMPT_A + " For comparisons, explicitly contrast each option's requirements and limitations before drawing a conclusion."

base = dict(temperature=0.2, k=5, prompt="A", retrieval=True, loop=True)
configs = [
    dict(name="Baseline", **base),
    dict(base, name="Temperature 0.7", temperature=0.7),
    dict(base, name="Temperature 1.2", temperature=1.2),
    dict(base, name="Top-k=3", k=3),
    dict(base, name="Top-k=10", k=10),
    dict(base, name="Prompt B", prompt="B"),
    dict(base, name="Without retrieval", retrieval=False),
    dict(base, name="Without reflection", loop=False),
]


FOLDER = ROOT / "data/evaluation/simple_study"
RESULTS_FILE = FOLDER / "results.json"
GRADES_FILE = FOLDER / "grading.csv"
display(pd.DataFrame(configs))
calls = REPEATS * len(questions) * sum(2 if c["loop"] else 1 for c in configs)
print(f"Planned: {len(configs) * REPEATS * len(questions)} answers; normally {calls} generation calls, plus embeddings and any retries.")


,name,temperature,k,prompt,retrieval,loop
0,Baseline,0.2,5,A,True,True
1,Temperature 0.7,0.7,5,A,True,True
2,Temperature 1.2,1.2,5,A,True,True
3,Top-k=3,0.2,3,A,True,True
4,Top-k=10,0.2,10,A,True,True
5,Prompt B,0.2,5,B,True,True
6,Without retrieval,0.2,5,A,False,True
7,Without reflection,0.2,5,A,True,False


Planned: 576 answers; normally 1080 generation calls, plus embeddings and any retries.


## 3. Run experiment

In [ ]:
# temporary set this experiment options
def ask(question, config, context):
    saved = (app.TEMPERATURE, app.SYSTEM_PROMPT, app.USE_REFLECTION, app.converse)
    calls = 0
    def counted_call(messages, **kwargs):
        nonlocal calls
        calls += 1
        reply = saved[3](messages, **kwargs)
        if not isinstance(reply, str) or not reply.strip():
            raise ValueError("The model returned an empty answer.")
        return reply
    try:
        app.TEMPERATURE = config["temperature"]
        app.SYSTEM_PROMPT = PROMPT_A if config["prompt"] == "A" else PROMPT_B
        app.USE_REFLECTION = config["loop"]
        app.converse = counted_call
        answer = app.answer(question["question"], context, history=[])
        return answer, calls
    finally:
        app.TEMPERATURE, app.SYSTEM_PROMPT, app.USE_REFLECTION, app.converse = saved


In [ ]:
def run_experiments():
    results = []
    collection = app.build_index(tuple(app.DOCUMENTS))
    for config in configs:
        for run in range(1, REPEATS + 1):
            for q in questions:
                context = app.retrieve(collection, q["question"], k=config["k"]) if config["retrieval"] else []
                answer, calls = ask(q, config, context)
                results.append(dict(config=config["name"], run=run, id=q["id"], type=q["type"],
                                    question=q["question"], answer=answer, context=context,
                                    source_indices=[source_index[t] for t in context], model_calls=calls))
            print(f"{config['name']} | run {run} complete", flush=True)
    FOLDER.mkdir(parents=True, exist_ok=True)
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"Saved {len(results)} answers to: {RESULTS_FILE}")
    return results


In [ ]:
# set to True to start the experiment
RUN_EXPERIMENTS = False
if RUN_EXPERIMENTS:
    results = run_experiments()
else:
    results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
    print(f"Study not started. Loaded {len(results)} saved answers.")


## 4. Scoring


In [13]:
results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
if results:
    gold = {q["id"]: q for q in questions}
    grades = pd.DataFrame(results)
    grades["reference_answer"] = grades["id"].map(lambda qid: gold[qid]["reference_answer"])
    grades["required_points"] = grades["id"].map(lambda qid: " | ".join(gold[qid]["required_points"]))
    grades[["correct", "grounding", "notes"]] = ""
    grades.to_csv(GRADES_FILE, index=False)
    print(f"Score {len(grades)} answers in: {GRADES_FILE}")
else:
    print("Run the experiments first; there are no answers to score yet.")


Score 576 answers in: /Users/nealonshah/Library/CloudStorage/OneDrive-IndianaUniversity/CSCI-B 490 (GenAI)/midterm-project/data/evaluation/simple_study/grading.csv


## 5. Display result

In [14]:
scored = pd.DataFrame()
if not RESULTS_FILE.exists() or not GRADES_FILE.exists():
    print("Generate and grade answers first. The results table will appear here.")
else:
    raw = pd.DataFrame(json.loads(RESULTS_FILE.read_text()))
    scores = pd.read_csv(GRADES_FILE, keep_default_na=False)
    scored = raw.merge(scores[["config", "run", "id", "correct", "grounding"]],
                       on=["config", "run", "id"], how="left", validate="one_to_one")
    for field in ["correct", "grounding"]:
        scored[field] = pd.to_numeric(scored[field].replace("", None), errors="raise")
        assert scored[field].dropna().isin([0, 1]).all(), f"{field} must be 0, 1, or blank."
    rows = []
    for config in configs:
        group = scored[scored["config"] == config["name"]]
        expected = len(questions) * REPEATS
        ready = (len(group) == expected and group["correct"].notna().all()
                 and set(group["run"]) == set(range(1, REPEATS + 1))
                 and all(set(g["id"]) == {q["id"] for q in questions} for _, g in group.groupby("run")))
        row = {"Configuration": config["name"], "Graded / expected": f"{group['correct'].notna().sum()} / {expected}",
               "Mean accuracy (%)": None, "STD (points)": None,
               "Grounding (%)": None, "Model calls / question": None}
        if ready:
            accuracy_by_run = group.groupby("run")["correct"].mean() * 100
            row["Mean accuracy (%)"] = accuracy_by_run.mean()
            row["STD (points)"] = accuracy_by_run.std(ddof=1)
            row["Model calls / question"] = group["model_calls"].mean()
            if config["retrieval"] and group["grounding"].notna().all():
                row["Grounding (%)"] = group["grounding"].mean() * 100
        rows.append(row)
    summary = pd.DataFrame(rows)
    display(summary.round(2))
    summary.to_csv(FOLDER / "summary.csv", index=False)


,Configuration,Graded / expected,Mean accuracy (%),STD (points),Grounding (%),Model calls / question
0,Baseline,72 / 72,50.00,0.00,98.61,2.0
1,Temperature 0.7,72 / 72,51.39,2.41,97.22,2.0
2,Temperature 1.2,72 / 72,48.61,2.41,98.61,2.0
3,Top-k=3,72 / 72,51.39,2.41,98.61,2.0
4,Top-k=10,72 / 72,58.33,4.17,94.44,2.0
5,Prompt B,72 / 72,50.00,0.00,98.61,2.0
6,Without retrieval,72 / 72,16.67,0.00,NaN,2.0
7,Without reflection,72 / 72,50.00,0.00,98.61,1.0
